CARGA DE DATOS Y GUARDADO

In [36]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Crear una sesión de Spark
spark = SparkSession.builder \
    .appName("Leer archivos desde HDFS") \
    .getOrCreate()


In [37]:
# Leer los archivos CSV desde HDFS
games = spark.read.csv("hdfs:///user/origen/data/games.csv", header=True, inferSchema=True)
teamstats = spark.read.csv("hdfs:///user/origen/data/teamstats.csv", header=True, inferSchema=True)
teams = spark.read.csv("hdfs:///user/origen/data/teams.csv", header=True, inferSchema=True)
shots = spark.read.csv("hdfs:///user/origen/data/shots.csv", header=True, inferSchema=True)
leagues = spark.read.csv("hdfs:///user/origen/data/leagues.csv", header=True, inferSchema=True)
players = spark.read.csv("hdfs:///user/origen/data/players.csv", header=True, inferSchema=True)
appearances = spark.read.csv("hdfs:///user/origen/data/appearances.csv", header=True, inferSchema=True)
teams_with_geo = spark.read.csv("hdfs:///user/origen/data/teams_with_geo.csv", header=True, inferSchema=True)

# Registrar los DataFrames como vistas temporales
games.createOrReplaceTempView("games")
leagues.createOrReplaceTempView("leagues")
shots.createOrReplaceTempView("shots")
players.createOrReplaceTempView("players")
teamstats.createOrReplaceTempView("teamstats")
teams.createOrReplaceTempView("teams")
appearances.createOrReplaceTempView("appearances")

In [38]:
# Obtengo la cantidad de filas y columnas de cada DataFrame
appearances_shape = (appearances.count(), len(appearances.columns))
games_shape = (games.count(), len(games.columns))
leagues_shape = (leagues.count(), len(leagues.columns))
shots_shape = (shots.count(), len(shots.columns))
teams_shape = (teams.count(), len(teams.columns))
teamstats_shape = (teamstats.count(), len(teamstats.columns))
players_shape = (players.count(), len(players.columns))

print("Appearances tiene", appearances_shape[0], "filas y", appearances_shape[1], "columnas")
print("Games tiene", games_shape[0], "filas y", games_shape[1], "columnas")
print("Leagues tiene", leagues_shape[0], "filas y", leagues_shape[1], "columnas")
print("Shots tiene", shots_shape[0], "filas y", shots_shape[1], "columnas")
print("Teams tiene", teams_shape[0], "filas y", teams_shape[1], "columnas")
print("Teamstats tiene", teamstats_shape[0], "filas y", teamstats_shape[1], "columnas")
print("Players tiene", players_shape[0], "filas y", players_shape[1], "columnas")


Appearances tiene 356513 filas y 19 columnas
Games tiene 12680 filas y 34 columnas
Leagues tiene 5 filas y 3 columnas
Shots tiene 324543 filas y 11 columnas
Teams tiene 146 filas y 2 columnas
Teamstats tiene 25360 filas y 16 columnas
Players tiene 7659 filas y 2 columnas


In [39]:
# Descripción de cada DataFrame
print("\nDescripción de appearances:")
appearances.describe().show()

print("\nDescripción de games:")
games.describe().show()

print("\nDescripción de leagues:")
leagues.describe().show()

print("\nDescripción de shots:")
shots.describe().show()

print("\nDescripción de teams:")
teams.describe().show()

print("\nDescripción de teamstats:")
teamstats.describe().show()

print("\nDescripción de players:")
players.describe().show()



Descripción de appearances:


+-------+-----------------+-----------------+-------------------+--------------------+------------------+-------------------+-------------------+-------------------+-------------------+------------------+-------------------+--------+-----------------+-------------------+--------------------+-----------------+------------------+-----------------+------------------+
|summary|           gameID|         playerID|              goals|            ownGoals|             shots|             xGoals|        xGoalsChain|      xGoalsBuildup|            assists|         keyPasses|           xAssists|position|    positionOrder|         yellowCard|             redCard|             time|      substituteIn|    substituteOut|          leagueID|
+-------+-----------------+-----------------+-------------------+--------------------+------------------+-------------------+-------------------+-------------------+-------------------+------------------+-------------------+--------+-----------------+---------------

+-------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+-------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+
|summary|           gameID|          leagueID|            season|        homeTeamID|        awayTeamID|         homeGoals|         awayGoals|    homeProbability|    drawProbability|    awayProbability| homeGoalsHalfTime| awayGoalsHalfTime|             B365H|             B365D|             B365A|               BWH|              BWD|               BWA|              

+-------+-----------------+------------------+------------------+-----------------+--------------+-----------+---------+-----------+-------------------+-------------------+-------------------+
|summary|           gameID|         shooterID|        assisterID|           minute|     situation| lastAction| shotType| shotResult|              xGoal|          positionX|          positionY|
+-------+-----------------+------------------+------------------+-----------------+--------------+-----------+---------+-----------+-------------------+-------------------+-------------------+
|  count|           324543|            324543|            324543|           324543|        324543|     324543|   324543|     324543|             324543|             324543|             324543|
|   mean|7832.533380784673|2691.2792110752657|2685.5154392815957| 48.5735880915626|          null|       null|     null|       null|0.10876032890816117| 0.8439679980167819| 0.5046133024674329|
| stddev|4729.926230146247|2272.984

+-------+-----------------+------------------+------------------+--------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------+
|summary|           gameID|            teamID|            season|location|             goals|            xGoals|             shots|     shotsOnTarget|              deep|              ppda|             fouls|           corners|       yellowCards|           redCards|result|
+-------+-----------------+------------------+------------------+--------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------+
|  count|            25360|             25360|             25360|   25360|             25360|             25360|             25360|             25360|             25360|            

In [40]:
# Eliminación de duplicados

In [53]:
# Lista de columnas con valores de apuestas faltantes
faltantes = ['B365H', 'B365D', 'B365A', 'BWH', 'BWD', 'BWA', 'IWH', 'IWD', 'IWA',
             'PSH', 'PSD', 'PSA', 'WHH', 'WHD', 'WHA', 'VCH', 'VCD', 'VCA',
             'PSCH', 'PSCD', 'PSCA']

# Reemplazar valores nulos en las columnas especificadas con -1
for col_name in faltantes:
    games = games.withColumn(col_name, col(col_name).cast("double"))

# Llenar valores nulos con -1 en todas las columnas
games_updated = games.fillna(-1)


In [58]:
# Aplicar la transformación para reemplazar los valores nulos y 'NA' en 'assisterID'
shots_updated = shots.withColumn("assisterID", when((col("assisterID").isNull()) | (col("assisterID") == 'NA'), -1).otherwise(col("assisterID")))

# Crear o reemplazar la vista temporal con el DataFrame transformado
shots_updated.createOrReplaceTempView("shots_transformed")

In [55]:

# Obtener tipos únicos de situaciones y últimas acciones
situations = shots.select("situation").distinct().rdd.flatMap(lambda x: x).collect()
last_actions = shots.select("lastAction").distinct().rdd.flatMap(lambda x: x).collect()

print("Tipos de situaciones:", situations)
print("Tipos de últimas acciones:", last_actions)

Tipos de situaciones: ['OpenPlay', 'FromCorner', 'DirectFreekick', 'SetPiece', 'Penalty']
Tipos de últimas acciones: ['Goal', 'Throughball', 'None', 'GoodSkill', 'HeadPass', 'Dispossessed', 'Punch', 'BallTouch', 'CrossNotClaimed', 'Save', 'LayOff', 'KeeperPickup', 'Challenge', 'PenaltyFaced', 'BlockedPass', 'Foul', 'OffsideProvoked', 'Clearance', 'Rebound', 'Card', 'End', 'SubstitutionOn', 'CornerAwarded', 'KeeperSweeper', 'ShieldBallOpp', 'BallRecovery', 'Cross', 'ChanceMissed', 'Start', 'TakeOn', 'Interception', 'OffsidePass', 'Error', 'FormationChange', 'Standard', 'Aerial', 'Chipped', 'Pass', 'Tackle', 'Smother', 'SubstitutionOff']


In [44]:
# Registrar los DataFrames como tablas temporales
teamstats.createOrReplaceTempView("teamstats")
appearances.createOrReplaceTempView("appearances")

# Consulta SQL para encontrar los registros en appearances correspondientes a los partidos con yellowCards nulos en teamstats
spark_total_amarillas = spark.sql("""
SELECT *
FROM appearances
WHERE gameID IN (
    SELECT gameID
    FROM teamstats
    WHERE yellowCards = 'NA'
)
""")

# Mostrar el resultado
spark_total_amarillas.show()


+------+--------+-----+--------+-----+------------------+------------------+------------------+-------+---------+------------------+--------+-------------+----------+-------+----+------------+-------------+--------+
|gameID|playerID|goals|ownGoals|shots|            xGoals|       xGoalsChain|     xGoalsBuildup|assists|keyPasses|          xAssists|position|positionOrder|yellowCard|redCard|time|substituteIn|substituteOut|leagueID|
+------+--------+-----+--------+-----+------------------+------------------+------------------+-------+---------+------------------+--------+-------------+----------+-------+----+------------+-------------+--------+
|  4888|    1825|    0|       0|    0|               0.0|               0.0|               0.0|      0|        0|               0.0|      GK|            1|         0|      0|  90|           0|            0|       2|
|  4888|    1438|    0|       0|    0|               0.0|               0.0|               0.0|      0|        0|               0.0|    

In [51]:
# Crear vistas temporales para los DataFrames teamstats y appearances
teamstats.createOrReplaceTempView("teamstats")
appearances.createOrReplaceTempView("appearances")

# Paso 1: Contar el número de tarjetas amarillas en appearances para gameID 4888 
spark.sql("""
CREATE OR REPLACE TEMP VIEW yellow_cards_count AS
SELECT gameID, COUNT(*) AS yellowCardsCount
FROM appearances
WHERE gameID = 4888 AND (yellowCard = 1 OR yellowCard = 2)
GROUP BY gameID
""")

# Paso 2: Actualizar la columna yellowCards en el DataFrame teamstats donde yellowCards era originalmente NA
teamstats_updated = spark.sql("""
SELECT 
    t.gameID, 
    t.teamID, 
    CASE 
        WHEN t.yellowCards IS NULL OR t.yellowCards = 'NA' THEN COALESCE(yc.yellowCardsCount, t.yellowCards)
        ELSE t.yellowCards
    END AS yellowCards
FROM 
    teamstats t
LEFT JOIN 
    yellow_cards_count yc 
ON 
    t.gameID = yc.gameID
""")

# Guardar el DataFrame como CSV
teamstats_updated.write.mode('overwrite').csv('ruta/donde/quieras/guardar/teamstats_updated_csv')


In [59]:

# Eliminar duplicados y asignar a nuevos DataFrames o vistas temporales
appearances = appearances.dropDuplicates()
games_updated = games_updated.dropDuplicates()
leagues = leagues.dropDuplicates()
players = players.dropDuplicates()
shots_updated = shots_updated.dropDuplicates()
teams = teams.dropDuplicates()
teamstats_updated = teamstats_updated.dropDuplicates()

# O bien, si prefieres utilizar SQL temporalmente, puedes hacerlo así:
# Crear vistas temporales para los DataFrames
appearances.createOrReplaceTempView("appearances")
games.createOrReplaceTempView("games")
leagues.createOrReplaceTempView("leagues")
players.createOrReplaceTempView("players")
shots.createOrReplaceTempView("shots")
teams.createOrReplaceTempView("teams")
teamstats_updated.createOrReplaceTempView("teamstats")

In [60]:
# Guardar el tratamiento de datos en carpeta 'Raw'
teamstats_updated.write.mode('overwrite').csv('Obligatorio/Limpios/teamstats.csv')
appearances.write.mode('overwrite').csv('Obligatorio/Limpios/appearances.csv')
games_updated.write.mode('overwrite').csv('Obligatorio/Limpios/games.csv')
leagues.write.mode('overwrite').csv('Obligatorio/Limpios/leagues.csv')
shots_updated.write.mode('overwrite').csv('Obligatorio/Limpios/shots_updated.csv')
teams.write.mode('overwrite').csv('Obligatorio/Limpios/teams.csv')
players.write.mode('overwrite').csv('Obligatorio/Limpios/players.csv')
teams_with_geo.write.mode('overwrite').csv('Obligatorio/Limpios/teams_with_geo.csv')